---
title: "Data Ingestion"
format:
    html:
        code-fold: false
---

## Overview

This notebook downloads the raw data for the courier assignment and bottleneck detection project. The study area is Washington, DC. We pull four open datasets. The drivable street network comes from OpenStreetMap, restaurant POIs come from OpenStreetMap, census tract demographics come from the DC ACS 5-Year tract layer, and 2024 annual average daily traffic (AADT) counts come from DDOT. We also pull ABCA liquor license locations as a secondary business licensing check on the OSM restaurant coverage. No public dataset of real delivery logs exists for DC, so delivery orders will be simulated downstream by pairing restaurants with demand locations drawn from census population density, consistent with the plan in the mid-point presentation. Everything lands in `data/raw`.

## Imports

In [1]:
import json
import time
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox
import pandas as pd
import requests

# paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# osmnx settings
ox.settings.use_cache = True
ox.settings.cache_folder = RAW / "osmnx_cache"
ox.settings.requests_timeout = 300

PLACE = "Washington, District of Columbia, USA"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW}")
print(f"osmnx {ox.__version__}, geopandas {gpd.__version__}, networkx {nx.__version__}")

Project root: /Users/komzic/Documents/GitHub/6400-final-project
Raw data dir: /Users/komzic/Documents/GitHub/6400-final-project/data/raw
osmnx 2.1.1, geopandas 1.1.3, networkx 3.6.1


## Road Network

In [2]:
# fetch the drivable street network for DC
t0 = time.time()
G = ox.graph_from_place(PLACE, network_type="drive")
print(f"Fetched in {time.time() - t0:.1f}s")
print(f"Nodes: {len(G.nodes):,}")
print(f"Edges: {len(G.edges):,}")

GRAPHML_RAW = RAW / "dc_drive_raw.graphml"
ox.save_graphml(G, GRAPHML_RAW)
print(f"Saved: {GRAPHML_RAW.name}")

Fetched in 10.1s
Nodes: 10,147
Edges: 27,105
Saved: dc_drive_raw.graphml


## Restaurant POIs

In [3]:
# fetch restaurant-type POIs from OSM
TAGS = {"amenity": ["restaurant", "fast_food", "cafe"]}
pois = ox.features_from_place(PLACE, TAGS)
pois = pois.reset_index()
keep = [c for c in ["element", "id", "name", "amenity", "cuisine", "geometry"] if c in pois.columns]
pois = pois[keep]
print(f"POIs fetched: {len(pois):,}")
print(pois["amenity"].value_counts().to_string())

POIS_RAW = RAW / "dc_restaurants_raw.geojson"
pois.to_file(POIS_RAW, driver="GeoJSON")
print(f"Saved: {POIS_RAW.name}")

POIs fetched: 2,042
amenity
restaurant    1071
fast_food      603
cafe           368
Saved: dc_restaurants_raw.geojson


## Census Tract Population

In [4]:
# paginated fetch helper for ArcGIS REST layers on the DC open data server
def fetch_arcgis_layer(url, out_fields="*", chunk=1000):
    features = []
    offset = 0
    while True:
        params = {
            "where": "1=1",
            "outFields": out_fields,
            "outSR": 4326,
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": chunk,
        }
        r = requests.get(f"{url}/query", params=params, timeout=120)
        r.raise_for_status()
        batch = r.json().get("features", [])
        features.extend(batch)
        if len(batch) < chunk:
            break
        offset += chunk
    return gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")

In [5]:
# ACS 5-year demographics by census tract, DP05_0001E is total population
ACS_URL = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Demographic_WebMercator/MapServer/38"
tracts = fetch_arcgis_layer(ACS_URL, out_fields="GEOID,NAMELSAD,ALAND,DP05_0001E")
tracts = tracts.rename(columns={"DP05_0001E": "population"})
print(f"Tracts: {len(tracts)}")
print(f"Total population: {tracts['population'].sum():,.0f}")

TRACTS_RAW = RAW / "dc_tracts_acs_raw.geojson"
tracts.to_file(TRACTS_RAW, driver="GeoJSON")
print(f"Saved: {TRACTS_RAW.name}")

Tracts: 206
Total population: 672,079
Saved: dc_tracts_acs_raw.geojson


## Traffic Volumes

In [6]:
# DDOT 2024 traffic volume segments with AADT counts
AADT_URL = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Transportation_TrafficVolume_WebMercator/MapServer/5"
aadt = fetch_arcgis_layer(AADT_URL, out_fields="ROUTEID,AADT,AADT_YEAR")
aadt = aadt.dropna(subset=["AADT"])
print(f"AADT segments: {len(aadt):,}")
print(f"AADT range: {aadt['AADT'].min():,.0f} to {aadt['AADT'].max():,.0f}")

AADT_RAW = RAW / "dc_aadt_2024_raw.geojson"
aadt.to_file(AADT_RAW, driver="GeoJSON")
print(f"Saved: {AADT_RAW.name}")

AADT segments: 8,324
AADT range: 81 to 243,160
Saved: dc_aadt_2024_raw.geojson


## Liquor License Locations

In [7]:
# ABCA liquor license locations, used as a secondary licensing check on OSM restaurant coverage
ABCA_URL = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/Business_Licensing_and_Grants_WebMercator/MapServer/5"
abca = fetch_arcgis_layer(ABCA_URL)
print(f"License locations: {len(abca):,}")

ABCA_RAW = RAW / "dc_abca_licenses_raw.geojson"
abca.to_file(ABCA_RAW, driver="GeoJSON")
print(f"Saved: {ABCA_RAW.name}")

License locations: 2,329
Saved: dc_abca_licenses_raw.geojson


## Raw Data Inventory

In [8]:
# summary of everything fetched
rows = [
    {"File": GRAPHML_RAW.name, "Records": f"{len(G.nodes):,} nodes / {len(G.edges):,} edges", "Size (MB)": round(GRAPHML_RAW.stat().st_size / 1e6, 1)},
    {"File": POIS_RAW.name, "Records": f"{len(pois):,} POIs", "Size (MB)": round(POIS_RAW.stat().st_size / 1e6, 1)},
    {"File": TRACTS_RAW.name, "Records": f"{len(tracts):,} tracts", "Size (MB)": round(TRACTS_RAW.stat().st_size / 1e6, 1)},
    {"File": AADT_RAW.name, "Records": f"{len(aadt):,} segments", "Size (MB)": round(AADT_RAW.stat().st_size / 1e6, 1)},
    {"File": ABCA_RAW.name, "Records": f"{len(abca):,} licenses", "Size (MB)": round(ABCA_RAW.stat().st_size / 1e6, 1)},
]
print(pd.DataFrame(rows).to_string(index=False))

                        File                     Records  Size (MB)
        dc_drive_raw.graphml 10,147 nodes / 27,105 edges       16.5
  dc_restaurants_raw.geojson                  2,042 POIs        0.5
   dc_tracts_acs_raw.geojson                  206 tracts        1.7
    dc_aadt_2024_raw.geojson              8,324 segments        5.0
dc_abca_licenses_raw.geojson              2,329 licenses        2.5


As we can see from the inventory above, all five raw datasets downloaded successfully and are small enough to hold comfortably in memory. The OSM network and POI pulls are the primary inputs. The ACS tracts provide the demand proxy, the AADT segments provide observed traffic volumes for travel time work, and the ABCA licenses give us an independent licensing dataset to sanity check restaurant coverage. Cleaning and refinement happen in the next notebook.